# OneDrive Duplicate Cleaner

Dieses Notebook findet doppelte Dateien per **SHA-256 Hash**, kann Duplikate löschen und kryptische Bildnamen anhand von Metadaten (EXIF) umbenennen.

Wichtige Sicherheit:
- Erst immer mit `DRY_RUN = True` testen.
- Erst danach `DELETE_DUPLICATES = True` setzen.
- Vor dem ersten echten Lauf ein Backup machen.


In [1]:
from __future__ import annotations


# Standardbibliotheken
import csv
import hashlib
import os
import re
import shutil
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional

# Pillow nutzen wir zum Auslesen von Bild-Metadaten (EXIF)
# Falls nicht installiert, läuft das Notebook trotzdem,
# aber EXIF-Datum kann dann nicht gelesen werden.
try:
    from PIL import Image
except ImportError:
    Image = None
    print("Hinweis: Pillow nicht installiert. EXIF-Auslesen ist deaktiviert. Installieren mit: pip install pillow")


In [2]:
# === Konfiguration ===
# Basisordner, der rekursiv gescannt wird.
# Standard: dein OneDrive-Ordner im Benutzerprofil.
ROOT_DIR = Path.home() / "OneDrive"   # Bei Bedarf anpassen

# Sicherheitsmodus:
# True  => nur Vorschau/Aktionsplanung, keine echten Aenderungen
# False => echte Aenderungen erlaubt (nur mit Bestaetigung unten)
DRY_RUN = True

# Aktionen aktivieren/deaktivieren
DELETE_DUPLICATES = False
RENAME_IMAGES = False
FIND_SIMILAR_IMAGES = True
FIND_AI_VISUAL_GROUPS = True
MOVE_AI_GROUPS_TO_QUARANTINE = False

# Zielordner fuer KI-Quarantaene
AI_QUARANTINE_DIR = ROOT_DIR / "_duplicate_cleaner_quarantine" / "ai_visual"

# Minilog-Einstellungen
VERBOSE = True
LOG_EVERY_FILES = 500      # Alle N Dateien einen Fortschrittslog
LOG_EVERY_BUCKETS = 50     # Alle N Groessen-Gruppen einen Fortschrittslog

# Vorschau-Einstellungen
SHOW_PREVIEW_DETAILS = True
PREVIEW_MAX_DUPLICATE_GROUPS = 0   # 0 oder negativ => alle Gruppen drucken
PREVIEW_MAX_RENAMES = 0            # 0 oder negativ => alle Umbenennungen drucken
PREVIEW_MAX_SIMILAR_GROUPS = 0     # 0 oder negativ => alle Aehnlichkeitsgruppen drucken
PREVIEW_MAX_AI_GROUPS = 0          # 0 oder negativ => alle KI-Gruppen drucken
PREVIEW_MAX_AI_QUARANTINE = 0      # 0 oder negativ => alle Quarantaene-Aktionen drucken

# Aehnlichkeits-Erkennung (Near-Duplicates, nicht exakt gleich)
SIMILAR_HASH_THRESHOLD = 6         # 0..64, kleiner = strenger
SIMILAR_NEIGHBOR_WINDOW = 20       # Vergleicht je Bild nur mit den naechsten N Bildern gleicher Aufloesung
SIMILAR_MIN_FILE_SIZE_BYTES = 20000

# KI-Erkennung fuer "gleiches Motiv mit kleinen Bewegungen"
AI_MODEL_NAME = "openai/clip-vit-base-patch32"
AI_SIMILARITY_THRESHOLD = 0.93     # Hoeher = strenger, 0.90-0.96 typischer Bereich
AI_NEIGHBOR_WINDOW = 60
AI_MIN_FILE_SIZE_BYTES = 20000
AI_BATCH_SIZE = 16

# Sicherheitsbremse fuer Live-Lauf
# Fuer echte Aenderungen MUSS alles passen:
# 1) DRY_RUN = False
# 2) CONFIRM_EXECUTION = True
# 3) CONFIRM_TEXT exakt wie REQUIRED_CONFIRM_TEXT
CONFIRM_EXECUTION = False
CONFIRM_TEXT = ""
REQUIRED_CONFIRM_TEXT = "JA_LOESCHEN_UND_UMBENENNEN"

# Verzeichnisregeln
FOLLOW_SYMLINKS = False
EXCLUDE_DIR_NAMES = {
    ".git", ".idea", "__pycache__", "$RECYCLE.BIN",
    "_duplicate_cleaner_reports", "_duplicate_cleaner_quarantine",
}

# Bilddateiendungen, fuer die Umbenennung/Aehnlichkeit geprueft wird
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff", ".heic", ".heif", ".bmp"}

# Optional: nur bestimmte Endungen auf Duplikate pruefen.
# Leer lassen = alle Dateien beruecksichtigen.
INCLUDE_EXTENSIONS = set()

print(f"ROOT_DIR = {ROOT_DIR}")
print(f"DRY_RUN = {DRY_RUN}, DELETE_DUPLICATES = {DELETE_DUPLICATES}, RENAME_IMAGES = {RENAME_IMAGES}")
print(f"FIND_SIMILAR_IMAGES = {FIND_SIMILAR_IMAGES}, FIND_AI_VISUAL_GROUPS = {FIND_AI_VISUAL_GROUPS}, MOVE_AI_GROUPS_TO_QUARANTINE = {MOVE_AI_GROUPS_TO_QUARANTINE}")
print(f"AI_QUARANTINE_DIR = {AI_QUARANTINE_DIR}")
print(f"VERBOSE = {VERBOSE}, LOG_EVERY_FILES = {LOG_EVERY_FILES}, LOG_EVERY_BUCKETS = {LOG_EVERY_BUCKETS}")
print(f"SHOW_PREVIEW_DETAILS = {SHOW_PREVIEW_DETAILS}")
print(f"SIMILAR_HASH_THRESHOLD = {SIMILAR_HASH_THRESHOLD}, AI_SIMILARITY_THRESHOLD = {AI_SIMILARITY_THRESHOLD}")
print(f"CONFIRM_EXECUTION = {CONFIRM_EXECUTION}, CONFIRM_TEXT gesetzt = {bool(CONFIRM_TEXT)}")


ROOT_DIR = C:\Users\Carina\OneDrive
DRY_RUN = True, DELETE_DUPLICATES = False, RENAME_IMAGES = False
FIND_SIMILAR_IMAGES = True, FIND_AI_VISUAL_GROUPS = True, MOVE_AI_GROUPS_TO_QUARANTINE = False
AI_QUARANTINE_DIR = C:\Users\Carina\OneDrive\_duplicate_cleaner_quarantine\ai_visual
VERBOSE = True, LOG_EVERY_FILES = 500, LOG_EVERY_BUCKETS = 50
SHOW_PREVIEW_DETAILS = True
SIMILAR_HASH_THRESHOLD = 6, AI_SIMILARITY_THRESHOLD = 0.93
CONFIRM_EXECUTION = False, CONFIRM_TEXT gesetzt = False


In [3]:
def iter_files(root: Path, exclude_dir_names: set[str], follow_symlinks: bool = False) -> Iterable[Path]:
    """Liefert alle Dateien unterhalb von root (rekursiv), ohne ausgeschlossene Ordner."""
    root = root.resolve()

    # os.walk liefert (aktueller Ordner, Unterordner, Dateien)
    for dirpath, dirnames, filenames in os.walk(root, topdown=True, followlinks=follow_symlinks):
        # Verzeichnisse frueh filtern, damit sie gar nicht erst betreten werden.
        dirnames[:] = [d for d in dirnames if d not in exclude_dir_names]

        base = Path(dirpath)
        for name in filenames:
            p = base / name
            if p.is_file():
                yield p


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Berechnet den SHA-256 Hash einer Datei in Bloecken (speicherschonend)."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def find_duplicate_groups(root: Path, include_extensions: set[str] | None = None) -> list[dict]:
    """
    Findet Duplikate in zwei Stufen:
    1) Gruppierung nach Dateigroesse (schneller Vorfilter)
    2) Innerhalb gleicher Groesse Vergleich per SHA-256
    """
    log(f"[SCAN] Starte Duplikat-Scan unter: {root}")

    size_buckets: Dict[int, List[Path]] = defaultdict(list)
    include_extensions = {e.lower() for e in (include_extensions or set())}

    scanned_files = 0
    skipped_by_extension = 0

    # Erstes Einsammeln: Dateien nach Groesse gruppieren
    for p in iter_files(root, EXCLUDE_DIR_NAMES, follow_symlinks=FOLLOW_SYMLINKS):
        scanned_files += 1
        if LOG_EVERY_FILES and scanned_files % LOG_EVERY_FILES == 0:
            log(f"[SCAN] Dateien gescannt: {scanned_files:,}")

        if include_extensions and p.suffix.lower() not in include_extensions:
            skipped_by_extension += 1
            continue

        try:
            size_buckets[p.stat().st_size].append(p)
        except OSError:
            # Datei kann z. B. waehrend des Scans verschwinden/gesperrt sein
            continue

    log(
        f"[SCAN] Fertig: {scanned_files:,} Dateien, "
        f"{len(size_buckets):,} Groessen-Gruppen, {skipped_by_extension:,} per Endung uebersprungen"
    )

    duplicate_groups: list[dict] = []
    candidate_buckets = [(size, paths) for size, paths in size_buckets.items() if len(paths) > 1]
    log(f"[HASH] Starte Hash-Vergleich fuer {len(candidate_buckets):,} Kandidatengruppen")

    hashed_files = 0
    for bucket_idx, (size, paths) in enumerate(candidate_buckets, start=1):
        if LOG_EVERY_BUCKETS and bucket_idx % LOG_EVERY_BUCKETS == 0:
            log(f"[HASH] Gruppe {bucket_idx:,}/{len(candidate_buckets):,} (Groesse={size} bytes, Dateien={len(paths)})")

        hash_buckets: Dict[str, List[Path]] = defaultdict(list)
        for p in paths:
            try:
                digest = sha256_file(p)
                hash_buckets[digest].append(p)
                hashed_files += 1
                if LOG_EVERY_FILES and hashed_files % LOG_EVERY_FILES == 0:
                    log(f"[HASH] Dateien gehasht: {hashed_files:,}")
            except OSError:
                continue

        # Jede Hash-Gruppe mit >=2 Dateien ist ein echtes Duplikat-Set
        for digest, dup_paths in hash_buckets.items():
            if len(dup_paths) < 2:
                continue

            # Regel fuer "behalten":
            # 1) aelteste Aenderung zuerst (kleinste mtime)
            # 2) bei Gleichstand kuerzerer Pfad
            # 3) dann alphabetisch
            sorted_paths = sorted(
                dup_paths,
                key=lambda x: (
                    x.stat().st_mtime if x.exists() else float("inf"),
                    len(str(x)),
                    str(x).lower(),
                ),
            )
            keep = sorted_paths[0]
            remove = sorted_paths[1:]

            duplicate_groups.append({
                "hash": digest,
                "size": size,
                "keep": keep,
                "remove": remove,
                "count": len(sorted_paths),
            })

    duplicate_files = sum(len(g["remove"]) for g in duplicate_groups)
    log(f"[DONE] Duplikatgruppen: {len(duplicate_groups):,}, zu entfernen: {duplicate_files:,}")

    return duplicate_groups


def build_duplicate_plan_actions(groups: list[dict]) -> list[dict]:
    """Erstellt eine detailierte Loesch-Vorschau pro Datei (ohne echte Aenderung)."""
    actions = []
    for idx, g in enumerate(groups, start=1):
        for p in g["remove"]:
            actions.append({
                "group": idx,
                "action": "delete",
                "path": str(p),
                "hash": g["hash"],
                "size": g["size"],
                "kept": str(g["keep"]),
                "status": "planned",
                "error": "",
            })
    return actions


def delete_duplicates(groups: list[dict], dry_run: bool = True) -> list[dict]:
    """Loescht alle als Duplikat markierten Dateien (ausser der Keep-Datei)."""
    total = sum(len(g["remove"]) for g in groups)
    mode = "DRY-RUN" if dry_run else "LIVE"
    log(f"[DELETE] Starte ({mode}), geplanter Umfang: {total:,} Dateien")

    actions = []
    processed = 0

    for g in groups:
        for p in g["remove"]:
            processed += 1
            action = {
                "action": "delete",
                "path": str(p),
                "hash": g["hash"],
                "size": g["size"],
                "kept": str(g["keep"]),
                "status": "planned" if dry_run else "deleted",
                "error": "",
            }

            if dry_run:
                log(f"[PLAN DELETE] {p}")
            else:
                log(f"[DELETE] {p}")
                try:
                    p.unlink()  # Datei wirklich entfernen
                except OSError as e:
                    action["status"] = "error"
                    action["error"] = str(e)
                    log(f"[DELETE ERROR] {p} -> {e}")

            if LOG_EVERY_FILES and processed % LOG_EVERY_FILES == 0:
                log(f"[DELETE] Fortschritt: {processed:,}/{total:,}")

            actions.append(action)

    log(f"[DELETE] Fertig, Aktionen: {len(actions):,}")
    return actions


In [4]:
# Formate, in denen EXIF-Datum typischerweise vorliegt
DATE_PATTERNS = [
    "%Y:%m:%d %H:%M:%S",
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%d",
]

# Typische kryptische Dateinamen: IMG_20230213_..., PXL_..., DSC..., UUIDs, nur Zahlen etc.
CRYPTIC_NAME_RE = re.compile(
    r"^(?:"
    r"(?:img|dsc|pxl|mvimg|wp_|screenshot|photo)[-_ ]?\d.*"
    r"|\d{6,}"
    r"|[a-f0-9]{16,}"
    r"|[a-z0-9_-]{12,}"
    r")$",
    re.IGNORECASE,
)

# Bereits sinnvolle Datumsnamen sollen nicht nochmal umbenannt werden
DATE_PREFIX_RE = re.compile(r"^\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}$")


def parse_exif_datetime(raw: str) -> Optional[datetime]:
    """Parst EXIF-Datumstexte in ein datetime-Objekt."""
    raw = (raw or "").strip()
    if not raw:
        return None

    for fmt in DATE_PATTERNS:
        try:
            return datetime.strptime(raw, fmt)
        except ValueError:
            pass

    return None


def get_image_datetime(path: Path) -> Optional[datetime]:
    """
    Liest bevorzugt das Aufnahmedatum aus EXIF:
    - DateTimeOriginal (36867)
    - DateTimeDigitized (36868)
    - DateTime (306)
    """
    if Image is None:
        return None

    try:
        with Image.open(path) as img:
            exif = getattr(img, "getexif", lambda: None)()
            if not exif:
                return None

            for tag_id in (36867, 36868, 306):
                raw = exif.get(tag_id)
                if raw:
                    dt = parse_exif_datetime(str(raw))
                    if dt:
                        return dt
    except Exception:
        # Defekte Datei / unbekanntes Format -> einfach ueberspringen
        return None

    return None


def is_cryptic_basename(stem: str) -> bool:
    """Heuristik, ob der Dateiname kryptisch wirkt und umbenannt werden sollte."""
    s = stem.strip()
    if not s:
        return False

    # Bereits im Ziel-Datumsformat -> nicht kryptisch
    if DATE_PREFIX_RE.match(s):
        return False

    # Treffer auf bekannte kryptische Muster
    if CRYPTIC_NAME_RE.match(s):
        return True

    # Zusatzregel: viele Ziffern, wenige Buchstaben
    letters = sum(ch.isalpha() for ch in s)
    digits = sum(ch.isdigit() for ch in s)
    if len(s) >= 8 and digits >= letters * 2:
        return True

    return False


def unique_path(target: Path) -> Path:
    """Erzeugt bei Namenskonflikt einen eindeutigen Dateinamen mit _1, _2, ..."""
    if not target.exists():
        return target

    stem, suffix = target.stem, target.suffix
    i = 1
    while True:
        candidate = target.with_name(f"{stem}_{i}{suffix}")
        if not candidate.exists():
            return candidate
        i += 1


def rename_cryptic_images(root: Path, dry_run: bool = True) -> list[dict]:
    """Benennt nur Bilddateien mit kryptischem Namen in Datumsnamen um."""
    mode = "DRY-RUN" if dry_run else "LIVE"
    log(f"[RENAME] Starte ({mode})")

    actions: list[dict] = []
    scanned_images = 0
    cryptic_candidates = 0

    for p in iter_files(root, EXCLUDE_DIR_NAMES, follow_symlinks=FOLLOW_SYMLINKS):
        ext = p.suffix.lower()
        if ext not in IMAGE_EXTENSIONS:
            continue

        scanned_images += 1
        if LOG_EVERY_FILES and scanned_images % LOG_EVERY_FILES == 0:
            log(f"[RENAME] Bilder geprueft: {scanned_images:,}")

        stem = p.stem
        if not is_cryptic_basename(stem):
            continue

        cryptic_candidates += 1

        # Bevorzugt EXIF-Datum nutzen, sonst Dateisystemzeit als Fallback
        dt = get_image_datetime(p)
        if dt is None:
            try:
                dt = datetime.fromtimestamp(p.stat().st_mtime)
            except OSError:
                continue

        new_stem = dt.strftime("%Y-%m-%d_%H-%M-%S")
        if stem == new_stem:
            continue

        target = unique_path(p.with_name(new_stem + ext))

        action = {
            "action": "rename",
            "from": str(p),
            "to": str(target),
            "status": "planned" if dry_run else "renamed",
            "error": "",
        }

        if dry_run:
            log(f"[PLAN RENAME] {p.name} -> {target.name}")
        else:
            log(f"[RENAME] {p.name} -> {target.name}")
            try:
                p.rename(target)
            except OSError as e:
                action["status"] = "error"
                action["error"] = str(e)
                log(f"[RENAME ERROR] {p} -> {e}")

        actions.append(action)

    log(
        f"[RENAME] Fertig: gepruefte Bilder={scanned_images:,}, "
        f"kryptische Kandidaten={cryptic_candidates:,}, Aktionen={len(actions):,}"
    )
    return actions


def dhash64_from_path(path: Path) -> Optional[int]:
    """Perceptual hash (dHash, 64-bit) fuer Aehnlichkeitsvergleich."""
    if Image is None:
        return None

    try:
        with Image.open(path) as img:
            gray = img.convert("L").resize((9, 8))
            pixels = list(gray.getdata())

        digest = 0
        bit = 0
        for y in range(8):
            row = y * 9
            for x in range(8):
                left_px = pixels[row + x]
                right_px = pixels[row + x + 1]
                if left_px > right_px:
                    digest |= (1 << bit)
                bit += 1

        return digest
    except Exception:
        return None


def hamming_distance_64(a: int, b: int) -> int:
    """Anzahl unterschiedlicher Bits zwischen zwei 64-bit Hashes."""
    return (a ^ b).bit_count()


def find_similar_image_groups(
    root: Path,
    threshold: int = 6,
    neighbor_window: int = 20,
    min_file_size_bytes: int = 0,
) -> dict:
    """
    Findet sehr aehnliche Bilder (Near-Duplicates) per dHash + Hamming-Distanz.

    Performance-Idee:
    - nur gleiche Aufloesung miteinander vergleichen
    - innerhalb gleicher Aufloesung nur mit den naechsten N Bildern (nach Zeit)
    """
    if Image is None:
        log("[SIMILAR] Pillow nicht verfuegbar -> Aehnlichkeitsanalyse uebersprungen")
        return {"groups": [], "actions": [], "pairs": []}

    threshold = max(0, min(64, int(threshold)))
    neighbor_window = max(1, int(neighbor_window))

    log(
        f"[SIMILAR] Starte (threshold={threshold}, window={neighbor_window}, min_size={min_file_size_bytes} bytes)"
    )

    signatures = []
    scanned_images = 0
    skipped_small = 0

    for p in iter_files(root, EXCLUDE_DIR_NAMES, follow_symlinks=FOLLOW_SYMLINKS):
        ext = p.suffix.lower()
        if ext not in IMAGE_EXTENSIONS:
            continue

        scanned_images += 1
        if LOG_EVERY_FILES and scanned_images % LOG_EVERY_FILES == 0:
            log(f"[SIMILAR] Bilder geprueft: {scanned_images:,}")

        try:
            stat = p.stat()
        except OSError:
            continue

        if stat.st_size < min_file_size_bytes:
            skipped_small += 1
            continue

        digest = dhash64_from_path(p)
        if digest is None:
            continue

        dt = get_image_datetime(p)
        ts = dt.timestamp() if dt else stat.st_mtime

        signatures.append(
            {
                "path": p,
                "hash": digest,
                "width": None,
                "height": None,
                "timestamp": ts,
                "mtime": stat.st_mtime,
            }
        )

    # Fuer Aufloesungs-Bucketing Bilddimensionen separat lesen
    prepared = []
    for item in signatures:
        p = item["path"]
        try:
            with Image.open(p) as img:
                w, h = img.size
        except Exception:
            continue

        item["width"] = w
        item["height"] = h
        prepared.append(item)

    signatures = prepared
    log(
        f"[SIMILAR] Signaturen erstellt: {len(signatures):,} "
        f"(gescannt={scanned_images:,}, zu_klein={skipped_small:,})"
    )

    buckets: Dict[tuple[int, int], List[dict]] = defaultdict(list)
    for s in signatures:
        buckets[(s["width"], s["height"])].append(s)

    pairs = []
    bucket_items = list(buckets.items())
    for b_idx, ((w, h), items) in enumerate(bucket_items, start=1):
        if len(items) < 2:
            continue

        if LOG_EVERY_BUCKETS and b_idx % LOG_EVERY_BUCKETS == 0:
            log(f"[SIMILAR] Bucket {b_idx:,}/{len(bucket_items):,}: {w}x{h}, Dateien={len(items):,}")

        items_sorted = sorted(items, key=lambda x: (x["timestamp"], str(x["path"]).lower()))

        for i, a in enumerate(items_sorted):
            end = min(len(items_sorted), i + 1 + neighbor_window)
            for j in range(i + 1, end):
                b = items_sorted[j]
                dist = hamming_distance_64(a["hash"], b["hash"])
                if dist <= threshold:
                    pairs.append(
                        {
                            "path_a": str(a["path"]),
                            "path_b": str(b["path"]),
                            "distance": dist,
                            "width": w,
                            "height": h,
                        }
                    )

    if not pairs:
        log("[SIMILAR] Keine aehnlichen Bilder gefunden")
        return {"groups": [], "actions": [], "pairs": []}

    # Union-Find fuer Gruppenbildung aus Paaren
    parent: Dict[str, str] = {}

    def uf_find(x: str) -> str:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def uf_union(a: str, b: str) -> None:
        ra = uf_find(a)
        rb = uf_find(b)
        if ra != rb:
            parent[rb] = ra

    for pair in pairs:
        a = pair["path_a"]
        b = pair["path_b"]
        if a not in parent:
            parent[a] = a
        if b not in parent:
            parent[b] = b
        uf_union(a, b)

    comps: Dict[str, List[str]] = defaultdict(list)
    for node in parent:
        comps[uf_find(node)].append(node)

    groups = []
    actions = []

    def mtime_safe(path_str: str) -> float:
        try:
            return Path(path_str).stat().st_mtime
        except OSError:
            return float("inf")

    for gid, paths in enumerate(sorted(comps.values(), key=lambda x: len(x), reverse=True), start=1):
        if len(paths) < 2:
            continue

        sorted_paths = sorted(paths, key=lambda s: (mtime_safe(s), len(s), s.lower()))
        keep = sorted_paths[0]
        similar = sorted_paths[1:]

        groups.append(
            {
                "group": gid,
                "keep": keep,
                "similar": similar,
                "count": len(sorted_paths),
            }
        )

        for p in similar:
            actions.append(
                {
                    "group": gid,
                    "action": "flag_similar",
                    "keep": keep,
                    "similar": p,
                    "status": "flagged",
                }
            )

    log(f"[SIMILAR] Fertig: Gruppen={len(groups):,}, markierte Dateien={len(actions):,}, Paare={len(pairs):,}")
    return {"groups": groups, "actions": actions, "pairs": pairs}


In [5]:
def log(message: str) -> None:
    """Kleiner Zeitstempel-Logger, steuerbar ueber VERBOSE."""
    if VERBOSE:
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}] {message}")


def should_execute_changes() -> bool:
    """Harte Sicherheitspruefung, bevor echte Dateiaenderungen passieren."""
    if DRY_RUN:
        print("Ausfuehrung blockiert: DRY_RUN=True")
        return False

    if not CONFIRM_EXECUTION:
        print("Ausfuehrung blockiert: CONFIRM_EXECUTION=False")
        return False

    if (CONFIRM_TEXT or "").strip() != REQUIRED_CONFIRM_TEXT:
        print("Ausfuehrung blockiert: CONFIRM_TEXT stimmt nicht mit REQUIRED_CONFIRM_TEXT ueberein.")
        print(f"Erwartet: {REQUIRED_CONFIRM_TEXT}")
        return False

    return True


def print_duplicate_plan(groups: list[dict], max_groups: Optional[int] = None) -> None:
    """Druckt den Vergleich: KEEP-Datei und zu loeschende Duplikate."""
    if not groups:
        print("Keine Duplikate gefunden.")
        return

    total_groups = len(groups)
    shown = groups
    if max_groups is not None and max_groups > 0:
        shown = groups[:max_groups]

    for idx, g in enumerate(shown, start=1):
        print(f"\n[Duplikat-Gruppe {idx}] Hash={g['hash'][:12]}... | Groesse={g['size']} bytes | Dateien={g['count']}")
        print(f"  BEHALTEN: {g['keep']}")
        for p in g["remove"]:
            print(f"  LOESCHEN : {p}")

    if len(shown) < total_groups:
        print(f"\n... {total_groups - len(shown)} weitere Gruppen nicht gedruckt (siehe CSV-Report).")


def print_rename_plan(actions: list[dict], max_rows: Optional[int] = None) -> None:
    """Druckt den Vergleich alter/neuer Bildnamen fuer die Vorschau."""
    if not actions:
        print("Keine Umbenennungen geplant.")
        return

    shown = actions
    if max_rows is not None and max_rows > 0:
        shown = actions[:max_rows]

    for idx, a in enumerate(shown, start=1):
        print(f"[{idx}] ALT: {a['from']}")
        print(f"    NEU: {a['to']}")

    if len(shown) < len(actions):
        print(f"... {len(actions) - len(shown)} weitere Umbenennungen nicht gedruckt (siehe CSV-Report).")


def print_similar_plan(groups: list[dict], max_groups: Optional[int] = None) -> None:
    """Druckt sehr aehnliche Bilder als Gruppen: REFERENZ + AEHNLICH."""
    if not groups:
        print("Keine aehnlichen Bildgruppen gefunden.")
        return

    total_groups = len(groups)
    shown = groups
    if max_groups is not None and max_groups > 0:
        shown = groups[:max_groups]

    for g in shown:
        print(f"\n[Aehnlichkeits-Gruppe {g['group']}] Dateien={g['count']}")
        print(f"  REFERENZ: {g['keep']}")
        for p in g["similar"]:
            print(f"  AEHNLICH: {p}")

    if len(shown) < total_groups:
        print(f"\n... {total_groups - len(shown)} weitere Gruppen nicht gedruckt (siehe CSV-Report).")


def save_csv(path: Path, rows: list[dict]) -> None:
    """Speichert Aktionslisten als CSV-Report."""
    if not rows:
        return

    fieldnames = sorted({k for row in rows for k in row.keys()})
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def summarize_duplicate_groups(groups: list[dict]) -> dict:
    """Berechnet eine kompakte Uebersicht ueber gefundene Duplikate."""
    total_groups = len(groups)
    total_duplicate_files = sum(len(g["remove"]) for g in groups)
    total_duplicate_bytes = sum(g["size"] * len(g["remove"]) for g in groups)

    return {
        "groups": total_groups,
        "duplicate_files": total_duplicate_files,
        "duplicate_bytes": total_duplicate_bytes,
        "duplicate_gb": round(total_duplicate_bytes / (1024**3), 3),
    }


def build_ai_quarantine_plan(ai_groups: list[dict], root_dir: Path, quarantine_dir: Path) -> list[dict]:
    """Plant Verschiebungen: pro KI-Gruppe 1 behalten, Rest nach Quarantaene."""
    actions = []

    for g in ai_groups:
        keep = g.get("keep")
        for src_str in g.get("similar", []):
            src = Path(src_str)
            if not src.exists():
                actions.append({
                    "group": g.get("group"),
                    "action": "move_to_quarantine",
                    "keep": keep,
                    "from": str(src),
                    "to": "",
                    "status": "missing",
                    "error": "source_missing",
                })
                continue

            try:
                rel = src.relative_to(root_dir)
            except ValueError:
                rel = Path(src.name)

            target = quarantine_dir / rel
            target = unique_path(target)

            actions.append({
                "group": g.get("group"),
                "action": "move_to_quarantine",
                "keep": keep,
                "from": str(src),
                "to": str(target),
                "status": "planned",
                "error": "",
            })

    return actions


def print_ai_quarantine_plan(actions: list[dict], max_rows: Optional[int] = None) -> None:
    """Druckt geplante Quarantaene-Verschiebungen."""
    if not actions:
        print("Keine KI-Quarantaene-Aktionen geplant.")
        return

    shown = actions
    if max_rows is not None and max_rows > 0:
        shown = actions[:max_rows]

    for idx, a in enumerate(shown, start=1):
        print(f"[{idx}] GRUPPE {a.get('group')} | BEHALTEN: {a.get('keep')}")
        print(f"    VERSCHIEBEN: {a.get('from')}")
        print(f"    NACH      : {a.get('to')}")

    if len(shown) < len(actions):
        print(f"... {len(actions) - len(shown)} weitere Quarantaene-Aktionen nicht gedruckt (siehe CSV-Report).")


def execute_ai_quarantine(actions: list[dict], dry_run: bool = True) -> list[dict]:
    """Fuehrt die geplanten Quarantaene-Verschiebungen aus."""
    mode = "DRY-RUN" if dry_run else "LIVE"
    log(f"[AI-QUAR] Starte ({mode}), geplanter Umfang: {len(actions):,} Dateien")

    out = []
    for idx, a in enumerate(actions, start=1):
        row = dict(a)

        if row.get("status") == "missing":
            out.append(row)
            continue

        src = Path(row["from"])
        dst = Path(row["to"])

        if dry_run:
            log(f"[AI-QUAR PLAN] {src} -> {dst}")
            out.append(row)
            continue

        try:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            row["status"] = "moved"
            log(f"[AI-QUAR MOVE] {src} -> {dst}")
        except Exception as e:
            row["status"] = "error"
            row["error"] = str(e)
            log(f"[AI-QUAR ERROR] {src} -> {e}")

        if LOG_EVERY_FILES and idx % LOG_EVERY_FILES == 0:
            log(f"[AI-QUAR] Fortschritt: {idx:,}/{len(actions):,}")

        out.append(row)

    log(f"[AI-QUAR] Fertig, Aktionen: {len(out):,}")
    return out


In [6]:
# 1) Duplikate finden + detailierte Vorschau erzeugen
# Diese Zelle macht noch keine Aenderungen an Dateien.
log("===== PHASE 1: DUPLIKATE SCANNEN =====")
groups = find_duplicate_groups(ROOT_DIR, INCLUDE_EXTENSIONS)
summary = summarize_duplicate_groups(groups)

duplicate_plan_actions = build_duplicate_plan_actions(groups)

print("Duplikat-Zusammenfassung:")
for k, v in summary.items():
    print(f"- {k}: {v}")
print(f"- geplanter Loeschumfang (Dateien): {len(duplicate_plan_actions)}")

if SHOW_PREVIEW_DETAILS:
    limit = None if PREVIEW_MAX_DUPLICATE_GROUPS <= 0 else PREVIEW_MAX_DUPLICATE_GROUPS
    print_duplicate_plan(groups, max_groups=limit)
else:
    print("Detailvorschau deaktiviert (SHOW_PREVIEW_DETAILS=False).")


[19:11:35] ===== PHASE 1: DUPLIKATE SCANNEN =====
[19:11:35] [SCAN] Starte Duplikat-Scan unter: C:\Users\Carina\OneDrive
[19:11:35] [SCAN] Dateien gescannt: 500
[19:11:35] [SCAN] Fertig: 905 Dateien, 859 Groessen-Gruppen, 0 per Endung uebersprungen
[19:11:35] [HASH] Starte Hash-Vergleich fuer 39 Kandidatengruppen
[19:11:35] [DONE] Duplikatgruppen: 29, zu entfernen: 32
Duplikat-Zusammenfassung:
- groups: 29
- duplicate_files: 32
- duplicate_bytes: 45591621
- duplicate_gb: 0.042
- geplanter Loeschumfang (Dateien): 32

[Duplikat-Gruppe 1] Hash=3a392c9b1719... | Groesse=2871542 bytes | Dateien=2
  BEHALTEN: C:\Users\Carina\OneDrive\Laptop\Carina\PhD\Master Certificate.pdf
  LOESCHEN : C:\Users\Carina\OneDrive\Laptop\Carina\Bewerbung\Abschluss  Master.pdf

[Duplikat-Gruppe 2] Hash=656fab2e4836... | Groesse=259904 bytes | Dateien=2
  BEHALTEN: C:\Users\Carina\OneDrive\NAS\Carina\Bewerbung\Mappe\Arbeitszeugnis bytes Coding.pdf
  LOESCHEN : C:\Users\Carina\OneDrive\Laptop\Carina\Bewerbung\Arbe

In [7]:
# 1B) Aehnliche Bilder (Near-Duplicates) finden und flaggen
log("===== PHASE 1B: AEHNLICHE BILDER FLAGGEN =====")

similar_groups = []
similar_plan_actions = []
similar_pairs = []

if FIND_SIMILAR_IMAGES:
    similar_result = find_similar_image_groups(
        ROOT_DIR,
        threshold=SIMILAR_HASH_THRESHOLD,
        neighbor_window=SIMILAR_NEIGHBOR_WINDOW,
        min_file_size_bytes=SIMILAR_MIN_FILE_SIZE_BYTES,
    )

    similar_groups = similar_result.get("groups", [])
    similar_plan_actions = similar_result.get("actions", [])
    similar_pairs = similar_result.get("pairs", [])

    print("Aehnlichkeits-Zusammenfassung:")
    print(f"- gruppen: {len(similar_groups)}")
    print(f"- markierte_dateien: {len(similar_plan_actions)}")
    print(f"- paare: {len(similar_pairs)}")

    if SHOW_PREVIEW_DETAILS:
        limit = None if PREVIEW_MAX_SIMILAR_GROUPS <= 0 else PREVIEW_MAX_SIMILAR_GROUPS
        print_similar_plan(similar_groups, max_groups=limit)
    else:
        print("Detailvorschau deaktiviert (SHOW_PREVIEW_DETAILS=False).")
else:
    print("FIND_SIMILAR_IMAGES=False -> Aehnlichkeitssuche deaktiviert.")


[19:11:35] ===== PHASE 1B: AEHNLICHE BILDER FLAGGEN =====
[19:11:35] [SIMILAR] Starte (threshold=6, window=20, min_size=20000 bytes)


C:\Users\Carina\AppData\Local\Temp\ipykernel_21436\3690343538.py:178: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(gray.getdata())


[19:11:50] [SIMILAR] Signaturen erstellt: 282 (gescannt=282, zu_klein=0)
[19:11:50] [SIMILAR] Fertig: Gruppen=35, markierte Dateien=69, Paare=126
Aehnlichkeits-Zusammenfassung:
- gruppen: 35
- markierte_dateien: 69
- paare: 126

[Aehnlichkeits-Gruppe 1] Dateien=8
  REFERENZ: C:\Users\Carina\OneDrive\NAS\Carina\Dateien und Texte\Zertifikate\THM-PRM4TUSKJB.png
  AEHNLICH: C:\Users\Carina\OneDrive\NAS\Carina\Dateien und Texte\Zertifikate\THM-RWL1ISDCZY.png
  AEHNLICH: C:\Users\Carina\OneDrive\NAS\Carina\Dateien und Texte\Zertifikate\THM-ANRGT1JXVX.png
  AEHNLICH: C:\Users\Carina\OneDrive\NAS\Carina\Dateien und Texte\Zertifikate\THM-C2JIFKJPYI.png
  AEHNLICH: C:\Users\Carina\OneDrive\NAS\Carina\Dateien und Texte\Zertifikate\THM-PDXPWY9IE3.png
  AEHNLICH: C:\Users\Carina\OneDrive\NAS\Carina\Dateien und Texte\Zertifikate\THM-VNPIRVWZ5M.png
  AEHNLICH: C:\Users\Carina\OneDrive\NAS\Carina\Dateien und Texte\Zertifikate\THM-RF8XGBJKX1.png
  AEHNLICH: C:\Users\Carina\OneDrive\NAS\Carina\Dateien u

In [8]:
# 1C) AI visual groups (same motif, minimal motion)
# This phase only marks files; it does not delete anything.

globals().setdefault("FIND_AI_VISUAL_GROUPS", True)
globals().setdefault("AI_MODEL_NAME", "openai/clip-vit-base-patch32")
globals().setdefault("AI_SIMILARITY_THRESHOLD", 0.93)
globals().setdefault("AI_SIMILARITY_PERCENT", None)  # Example: 93.0
globals().setdefault("AI_NEIGHBOR_WINDOW", 60)
globals().setdefault("AI_MIN_FILE_SIZE_BYTES", 20000)
globals().setdefault("AI_BATCH_SIZE", 16)
globals().setdefault("AI_VERBOSE", True)
globals().setdefault("AI_VERBOSE_PAIR_LOG_LIMIT", 25)
globals().setdefault("PREVIEW_MAX_AI_GROUPS", 0)

import numpy as np


def _ai_print_groups(groups: list[dict], max_groups: Optional[int] = None) -> None:
    if not groups:
        print("No AI visual groups found.")
        return

    shown = groups
    if max_groups is not None and max_groups > 0:
        shown = groups[:max_groups]

    for g in shown:
        print(f"\n[AI Group {g['group']}] files={g['count']}")
        print(f"  KEEP (suggested): {g['keep']}")
        for s in g["similar"]:
            print(f"  VERY SIMILAR    : {s}")

    if len(shown) < len(groups):
        print(f"\n... {len(groups) - len(shown)} more AI groups not printed (see CSV report).")


def _load_clip_model(model_name: str):
    try:
        import torch
        from transformers import CLIPModel, CLIPProcessor
    except Exception:
        log("[AI] Missing packages. Install with: pip install torch transformers")
        return None, None, None

    try:
        model = CLIPModel.from_pretrained(model_name)
        processor = CLIPProcessor.from_pretrained(model_name)
    except Exception as e:
        log(f"[AI] Failed to load model ({model_name}): {e}")
        return None, None, None

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    log(f"[AI] CLIP loaded: {model_name} on {device}")
    return model, processor, torch


def _resolve_similarity_threshold(
    similarity_threshold: float,
    similarity_percent: Optional[float],
) -> float:
    if similarity_percent is not None:
        threshold = float(similarity_percent) / 100.0
    else:
        threshold = float(similarity_threshold)
        if threshold > 1.0:
            threshold = threshold / 100.0

    return max(0.0, min(1.0, threshold))


def find_ai_visual_groups(
    root: Path,
    similarity_threshold: float = 0.93,
    similarity_percent: Optional[float] = None,
    neighbor_window: int = 60,
    min_file_size_bytes: int = 0,
    model_name: str = "openai/clip-vit-base-patch32",
    batch_size: int = 16,
    verbose: bool = False,
    verbose_pair_log_limit: int = 0,
) -> dict:
    """
    Find burst-like images with very similar motifs via CLIP embeddings.
    Typical use case: repeated shutter shots with minimal movement.
    """
    if Image is None:
        log("[AI] Pillow missing -> AI visual grouping skipped")
        return {"groups": [], "actions": [], "pairs": []}

    model, processor, torch_mod = _load_clip_model(model_name)
    if model is None:
        return {"groups": [], "actions": [], "pairs": []}

    similarity_threshold = _resolve_similarity_threshold(similarity_threshold, similarity_percent)
    neighbor_window = max(1, int(neighbor_window))
    batch_size = max(1, int(batch_size))
    verbose_pair_log_limit = max(0, int(verbose_pair_log_limit))

    log(
        f"[AI] Starting visual grouping "
        f"(threshold={similarity_threshold:.4f} / {similarity_threshold * 100:.1f}%, "
        f"window={neighbor_window}, min_size={min_file_size_bytes}, verbose={verbose})"
    )

    candidates = []
    scan_stats = {
        "all_files": 0,
        "non_images": 0,
        "image_files": 0,
        "stat_errors": 0,
        "too_small": 0,
        "candidates": 0,
    }

    current_folder = None
    for p in iter_files(root, EXCLUDE_DIR_NAMES, follow_symlinks=FOLLOW_SYMLINKS):
        scan_stats["all_files"] += 1

        folder = str(p.parent)
        if verbose and folder != current_folder:
            current_folder = folder
            log(f"[AI][SCAN] Checking folder: {folder}")

        if p.suffix.lower() not in IMAGE_EXTENSIONS:
            scan_stats["non_images"] += 1
            continue

        scan_stats["image_files"] += 1
        if LOG_EVERY_FILES and scan_stats["image_files"] % LOG_EVERY_FILES == 0:
            log(f"[AI][SCAN] Image files checked: {scan_stats['image_files']:,}")

        try:
            st = p.stat()
        except OSError:
            scan_stats["stat_errors"] += 1
            if verbose:
                log(f"[AI][SCAN] Failed to stat file: {p}")
            continue

        if st.st_size < min_file_size_bytes:
            scan_stats["too_small"] += 1
            continue

        dt = get_image_datetime(p)
        ts = dt.timestamp() if dt else st.st_mtime
        candidates.append({"path": p, "timestamp": ts, "size": st.st_size, "mtime": st.st_mtime})
        scan_stats["candidates"] += 1

    if verbose:
        log(
            "[AI][SCAN] Summary: "
            f"files={scan_stats['all_files']:,}, "
            f"images={scan_stats['image_files']:,}, "
            f"candidates={scan_stats['candidates']:,}, "
            f"too_small={scan_stats['too_small']:,}, "
            f"stat_errors={scan_stats['stat_errors']:,}"
        )

    if len(candidates) < 2:
        log("[AI] Too few image candidates")
        return {"groups": [], "actions": [], "pairs": []}

    device = next(model.parameters()).device
    rows = []

    def _unwrap_clip_features(raw):
        if hasattr(raw, "image_embeds") and raw.image_embeds is not None:
            return raw.image_embeds
        if hasattr(raw, "pooler_output") and raw.pooler_output is not None:
            return raw.pooler_output
        if hasattr(raw, "last_hidden_state") and raw.last_hidden_state is not None:
            hidden = raw.last_hidden_state
            if getattr(hidden, "ndim", 0) >= 3:
                return hidden[:, 0, :]
            return hidden
        if isinstance(raw, (tuple, list)) and raw:
            return raw[0]
        return raw

    total_batches = (len(candidates) + batch_size - 1) // batch_size
    decode_errors_total = 0

    for batch_idx, start in enumerate(range(0, len(candidates), batch_size), start=1):
        chunk = candidates[start:start + batch_size]
        if verbose:
            log(f"[AI][EMBED] Batch {batch_idx}/{total_batches}: loading {len(chunk)} candidates")

        imgs = []
        meta = []
        decode_errors_batch = 0

        for row in chunk:
            try:
                with Image.open(row["path"]) as im:
                    imgs.append(im.convert("RGB"))
                meta.append(row)
            except Exception:
                decode_errors_batch += 1
                continue

        if decode_errors_batch:
            decode_errors_total += decode_errors_batch
            if verbose:
                log(f"[AI][EMBED] Batch {batch_idx}: decode errors={decode_errors_batch}")

        if not meta:
            if verbose:
                log(f"[AI][EMBED] Batch {batch_idx}: no valid images after decode")
            continue

        inputs = processor(images=imgs, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch_mod.no_grad():
            feats_raw = model.get_image_features(**inputs)
            feats = _unwrap_clip_features(feats_raw)

        if feats is None or not hasattr(feats, "norm"):
            log(
                f"[AI][EMBED] Unexpected feature output ({type(feats).__name__}) "
                f"for batch {batch_idx}; skipping batch"
            )
            continue

        feats = feats / feats.norm(dim=-1, keepdim=True).clamp_min(1e-12)
        arr = feats.detach().cpu().numpy().astype(np.float32)
        for m, emb in zip(meta, arr):
            r = dict(m)
            r["embedding"] = emb
            rows.append(r)

        if verbose:
            log(f"[AI][EMBED] Batch {batch_idx}: embeddings added={len(meta)}")

    if verbose:
        log(f"[AI][EMBED] Summary: embeddings={len(rows):,}, decode_errors={decode_errors_total:,}")

    if len(rows) < 2:
        log("[AI] Too few valid embeddings")
        return {"groups": [], "actions": [], "pairs": []}

    rows = sorted(rows, key=lambda r: (r["timestamp"], str(r["path"]).lower()))

    pairs = []
    comparisons = 0
    if verbose:
        log(f"[AI][PAIR] Starting pair search over {len(rows):,} embeddings")

    for i, a in enumerate(rows):
        if LOG_EVERY_FILES and i > 0 and i % LOG_EVERY_FILES == 0:
            log(f"[AI][PAIR] Progress: {i:,}/{len(rows):,}")

        end = min(len(rows), i + 1 + neighbor_window)
        for j in range(i + 1, end):
            b = rows[j]
            comparisons += 1
            sim = float(np.dot(a["embedding"], b["embedding"]))
            if sim >= similarity_threshold:
                pair = {
                    "path_a": str(a["path"]),
                    "path_b": str(b["path"]),
                    "similarity": round(sim, 5),
                }
                pairs.append(pair)

                if verbose and verbose_pair_log_limit > 0 and len(pairs) <= verbose_pair_log_limit:
                    log(
                        f"[AI][PAIR] match#{len(pairs)} sim={pair['similarity']:.5f} "
                        f"| {pair['path_a']} <-> {pair['path_b']}"
                    )

    if verbose:
        log(f"[AI][PAIR] Summary: comparisons={comparisons:,}, pairs={len(pairs):,}")

    if not pairs:
        log("[AI] No AI-similar pairs found")
        return {"groups": [], "actions": [], "pairs": []}

    parent: Dict[str, str] = {}

    def uf_find(x: str) -> str:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def uf_union(a: str, b: str) -> None:
        ra = uf_find(a)
        rb = uf_find(b)
        if ra != rb:
            parent[rb] = ra

    for pair in pairs:
        a = pair["path_a"]
        b = pair["path_b"]
        if a not in parent:
            parent[a] = a
        if b not in parent:
            parent[b] = b
        uf_union(a, b)

    components: Dict[str, List[str]] = defaultdict(list)
    for node in parent:
        components[uf_find(node)].append(node)

    row_by_path = {str(r["path"]): r for r in rows}

    def keep_key(path_str: str):
        # Keep suggestion: larger file first, then older mtime, then shorter path.
        rr = row_by_path.get(path_str)
        if rr is None:
            return (float("inf"), float("inf"), len(path_str), path_str.lower())
        return (-rr.get("size", 0), rr.get("mtime", float("inf")), len(path_str), path_str.lower())

    groups = []
    actions = []

    for gid, paths in enumerate(sorted(components.values(), key=lambda g: len(g), reverse=True), start=1):
        if len(paths) < 2:
            continue

        sorted_paths = sorted(paths, key=keep_key)
        keep = sorted_paths[0]
        similar = sorted_paths[1:]

        groups.append({
            "group": gid,
            "keep": keep,
            "similar": similar,
            "count": len(sorted_paths),
        })

        for s in similar:
            actions.append({
                "group": gid,
                "action": "flag_ai_similar",
                "keep": keep,
                "similar": s,
                "status": "flagged",
            })

    log(f"[AI] Done: groups={len(groups):,}, flagged_files={len(actions):,}, pairs={len(pairs):,}")
    return {"groups": groups, "actions": actions, "pairs": pairs}


log("===== PHASE 1C: AI VISUAL GROUPING =====")
ai_visual_groups = []
ai_visual_plan_actions = []
ai_visual_pairs = []

find_ai_visual_groups_enabled = bool(globals().get("FIND_AI_VISUAL_GROUPS", True))
root_for_ai = globals().get("ROOT_DIR")
if root_for_ai is None:
    root_for_ai = Path.cwd()
    log(f"[AI] ROOT_DIR not defined, using current directory: {root_for_ai}")

ai_similarity_threshold = globals().get("AI_SIMILARITY_THRESHOLD", 0.93)
ai_similarity_percent = globals().get("AI_SIMILARITY_PERCENT", None)
ai_neighbor_window = globals().get("AI_NEIGHBOR_WINDOW", 60)
ai_min_file_size_bytes = globals().get("AI_MIN_FILE_SIZE_BYTES", 0)
ai_model_name = globals().get("AI_MODEL_NAME", "openai/clip-vit-base-patch32")
ai_batch_size = globals().get("AI_BATCH_SIZE", 16)
ai_verbose = bool(globals().get("AI_VERBOSE", False))
ai_verbose_pair_log_limit = globals().get("AI_VERBOSE_PAIR_LOG_LIMIT", 0)
show_preview_details = bool(globals().get("SHOW_PREVIEW_DETAILS", True))
preview_max_ai_groups = globals().get("PREVIEW_MAX_AI_GROUPS", 0)

if find_ai_visual_groups_enabled:
    ai_result = find_ai_visual_groups(
        root_for_ai,
        similarity_threshold=ai_similarity_threshold,
        similarity_percent=ai_similarity_percent,
        neighbor_window=ai_neighbor_window,
        min_file_size_bytes=ai_min_file_size_bytes,
        model_name=ai_model_name,
        batch_size=ai_batch_size,
        verbose=ai_verbose,
        verbose_pair_log_limit=ai_verbose_pair_log_limit,
    )

    ai_visual_groups = ai_result.get("groups", [])
    ai_visual_plan_actions = ai_result.get("actions", [])
    ai_visual_pairs = ai_result.get("pairs", [])

    print("AI visual summary:")
    print(f"- groups: {len(ai_visual_groups)}")
    print(f"- flagged_files: {len(ai_visual_plan_actions)}")
    print(f"- pairs: {len(ai_visual_pairs)}")

    if show_preview_details:
        limit = None if preview_max_ai_groups <= 0 else preview_max_ai_groups
        _ai_print_groups(ai_visual_groups, max_groups=limit)
    else:
        print("Detail preview disabled (SHOW_PREVIEW_DETAILS=False).")
else:
    print("FIND_AI_VISUAL_GROUPS=False -> AI visual grouping disabled.")


[19:11:50] ===== PHASE 1C: AI VISUAL GROUPING =====


X:\Privat\Dublicatefinder (Onedrive)\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1256.24it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[19:11:58] [AI] CLIP loaded: openai/clip-vit-base-patch32 on cpu
[19:11:58] [AI] Starting visual grouping (threshold=0.9300 / 93.0%, window=60, min_size=20000, verbose=True)
[19:11:58] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive
[19:11:58] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\Bewerbung
[19:11:58] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\MBA
[19:11:58] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\MBA\new\Additional Documents
[19:11:58] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\MBA\new\Humanized Version
[19:11:58] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\MBA\new\Original Version
[19:11:58] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\PhD
[19:11:58] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\Wallpaper
[19:11:58] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Work\BBA\Anwesenheit\IT0227
[19:11:58

In [9]:
# 1D) KI-Gruppen in Quarantaene verschieben (1 behalten, Rest verschieben)
log("===== PHASE 1D: KI QUARANTAENE =====")

ai_quarantine_plan_actions = []
ai_quarantine_exec_actions = []

ai_groups = globals().get("ai_visual_groups", [])
if not ai_groups:
    print("Keine KI-Gruppen vorhanden. Bitte zuerst PHASE 1C laufen lassen.")
else:
    ai_quarantine_plan_actions = build_ai_quarantine_plan(ai_groups, ROOT_DIR, AI_QUARANTINE_DIR)
    print(f"Geplante KI-Quarantaene-Aktionen: {len(ai_quarantine_plan_actions)}")

    if SHOW_PREVIEW_DETAILS:
        limit = None if PREVIEW_MAX_AI_QUARANTINE <= 0 else PREVIEW_MAX_AI_QUARANTINE
        print_ai_quarantine_plan(ai_quarantine_plan_actions, max_rows=limit)
    else:
        print("Detailvorschau deaktiviert (SHOW_PREVIEW_DETAILS=False).")

    if MOVE_AI_GROUPS_TO_QUARANTINE:
        print("KI-Quarantaene-Ausfuehrung angefordert.")
        if should_execute_changes():
            ai_quarantine_exec_actions = execute_ai_quarantine(ai_quarantine_plan_actions, dry_run=False)
            moved = sum(1 for r in ai_quarantine_exec_actions if r.get('status') == 'moved')
            errs = sum(1 for r in ai_quarantine_exec_actions if r.get('status') == 'error')
            print(f"Quarantaene ausgefuehrt: moved={moved}, error={errs}")
        else:
            print("KI-Quarantaene-Ausfuehrung wurde aus Sicherheitsgruenden nicht gestartet.")
    else:
        print("MOVE_AI_GROUPS_TO_QUARANTINE=False -> Nur Vorschau, keine Verschiebung.")


[19:12:41] ===== PHASE 1D: KI QUARANTAENE =====
Geplante KI-Quarantaene-Aktionen: 157
[1] GRUPPE 1 | BEHALTEN: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041433336.jpg
    VERSCHIEBEN: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041704275.jpg
    NACH      : C:\Users\Carina\OneDrive\_duplicate_cleaner_quarantine\ai_visual\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041704275.jpg
[2] GRUPPE 1 | BEHALTEN: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041433336.jpg
    VERSCHIEBEN: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041722815.jpg
    NACH      : C:\Users\Carina\OneDrive\_duplicate_cleaner_quarantine\ai_visual\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041722815.jpg
[3] GRUPPE 1 | BEHALTEN: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041433336.jpg
    VERSCHIEBEN: C:\Users\Ca

In [10]:
# 2) Duplikate loeschen (nur nach Vorschau + Bestaetigung)
log("===== PHASE 2: DUPLIKATE LOESCHEN =====")

delete_actions = []

if DELETE_DUPLICATES:
    print("Delete-Ausfuehrung angefordert.")
    if should_execute_changes():
        delete_actions = delete_duplicates(groups, dry_run=False)
        print(f"Delete-Aktionen ausgefuehrt: {len(delete_actions)}")
    else:
        print("Delete-Ausfuehrung wurde aus Sicherheitsgruenden nicht gestartet.")
else:
    print("DELETE_DUPLICATES=False -> Keine Dateien geloescht.")


[19:12:41] ===== PHASE 2: DUPLIKATE LOESCHEN =====
DELETE_DUPLICATES=False -> Keine Dateien geloescht.


In [11]:
# 3) Kryptische Bildnamen: erst Vorschau, dann optional ausfuehren
log("===== PHASE 3: BILDER UMBENENNEN =====")

rename_preview_actions = []
rename_actions = []

if RENAME_IMAGES:
    # Immer zuerst Vorschau, unabhaengig von DRY_RUN
    rename_preview_actions = rename_cryptic_images(ROOT_DIR, dry_run=True)
    print(f"Geplante Umbenennungen: {len(rename_preview_actions)}")

    if SHOW_PREVIEW_DETAILS:
        limit = None if PREVIEW_MAX_RENAMES <= 0 else PREVIEW_MAX_RENAMES
        print_rename_plan(rename_preview_actions, max_rows=limit)
    else:
        print("Detailvorschau deaktiviert (SHOW_PREVIEW_DETAILS=False).")

    if should_execute_changes():
        rename_actions = rename_cryptic_images(ROOT_DIR, dry_run=False)
        print(f"Umbenennungen ausgefuehrt: {len(rename_actions)}")
    else:
        print("Rename-Ausfuehrung nicht gestartet (nur Vorschau).")
else:
    print("RENAME_IMAGES=False -> Keine Bilder umbenannt.")


[19:12:41] ===== PHASE 3: BILDER UMBENENNEN =====
RENAME_IMAGES=False -> Keine Bilder umbenannt.


In [12]:
# 5) Reports speichern
# CSV-Reports landen in einem eigenen Unterordner im ROOT_DIR.
log("===== PHASE 5: REPORTS SCHREIBEN =====")
report_dir = ROOT_DIR / "_duplicate_cleaner_reports"
report_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# Fallbacks, falls vorherige Phasen noch nicht liefen
duplicate_plan_actions = globals().get("duplicate_plan_actions", [])
rename_preview_actions = globals().get("rename_preview_actions", [])
similar_plan_actions = globals().get("similar_plan_actions", [])
similar_pairs = globals().get("similar_pairs", [])
delete_actions = globals().get("delete_actions", [])
rename_actions = globals().get("rename_actions", [])

dupe_plan_path = report_dir / f"duplicates_plan_{ts}.csv"
rename_plan_path = report_dir / f"renames_plan_{ts}.csv"
similar_plan_path = report_dir / f"similar_images_plan_{ts}.csv"
similar_pairs_path = report_dir / f"similar_images_pairs_{ts}.csv"
dupe_exec_path = report_dir / f"duplicates_executed_{ts}.csv"
rename_exec_path = report_dir / f"renames_executed_{ts}.csv"

if duplicate_plan_actions:
    save_csv(dupe_plan_path, duplicate_plan_actions)
    print(f"Duplikat-PLAN-Report: {dupe_plan_path}")
else:
    print("Kein Duplikat-PLAN-Report geschrieben (keine geplanten Loeschungen).")

if rename_preview_actions:
    save_csv(rename_plan_path, rename_preview_actions)
    print(f"Rename-PLAN-Report: {rename_plan_path}")
else:
    print("Kein Rename-PLAN-Report geschrieben (keine geplanten Umbenennungen).")

if similar_plan_actions:
    save_csv(similar_plan_path, similar_plan_actions)
    print(f"Similar-PLAN-Report: {similar_plan_path}")
else:
    print("Kein Similar-PLAN-Report geschrieben (keine aehnlichen Bilder markiert).")

if similar_pairs:
    save_csv(similar_pairs_path, similar_pairs)
    print(f"Similar-PAIR-Report: {similar_pairs_path}")
else:
    print("Kein Similar-PAIR-Report geschrieben (keine Paare gefunden).")

if delete_actions:
    save_csv(dupe_exec_path, delete_actions)
    print(f"Duplikat-EXEC-Report: {dupe_exec_path}")
else:
    print("Kein Duplikat-EXEC-Report geschrieben (nichts ausgefuehrt).")

if rename_actions:
    save_csv(rename_exec_path, rename_actions)
    print(f"Rename-EXEC-Report: {rename_exec_path}")
else:
    print("Kein Rename-EXEC-Report geschrieben (nichts ausgefuehrt).")


[19:12:41] ===== PHASE 5: REPORTS SCHREIBEN =====
Duplikat-PLAN-Report: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\duplicates_plan_20260301_191241.csv
Kein Rename-PLAN-Report geschrieben (keine geplanten Umbenennungen).
Similar-PLAN-Report: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\similar_images_plan_20260301_191241.csv
Similar-PAIR-Report: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\similar_images_pairs_20260301_191241.csv
Kein Duplikat-EXEC-Report geschrieben (nichts ausgefuehrt).
Kein Rename-EXEC-Report geschrieben (nichts ausgefuehrt).


In [13]:
# 6B) KI-Visual- und Quarantaene-Reports speichern (zusaetzlich)
log("===== PHASE 6B: KI REPORTS SCHREIBEN =====")

report_dir = ROOT_DIR / "_duplicate_cleaner_reports"
report_dir.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

ai_visual_plan_actions = globals().get("ai_visual_plan_actions", [])
ai_visual_pairs = globals().get("ai_visual_pairs", [])
ai_quarantine_plan_actions = globals().get("ai_quarantine_plan_actions", [])
ai_quarantine_exec_actions = globals().get("ai_quarantine_exec_actions", [])

ai_plan_path = report_dir / f"ai_visual_groups_plan_{ts}.csv"
ai_pairs_path = report_dir / f"ai_visual_groups_pairs_{ts}.csv"
ai_quar_plan_path = report_dir / f"ai_quarantine_plan_{ts}.csv"
ai_quar_exec_path = report_dir / f"ai_quarantine_executed_{ts}.csv"

if ai_visual_plan_actions:
    save_csv(ai_plan_path, ai_visual_plan_actions)
    print(f"AI-Visual-PLAN-Report: {ai_plan_path}")
else:
    print("Kein AI-Visual-PLAN-Report geschrieben (keine KI-Gruppen markiert).")

if ai_visual_pairs:
    save_csv(ai_pairs_path, ai_visual_pairs)
    print(f"AI-Visual-PAIR-Report: {ai_pairs_path}")
else:
    print("Kein AI-Visual-PAIR-Report geschrieben (keine KI-Paare gefunden).")

if ai_quarantine_plan_actions:
    save_csv(ai_quar_plan_path, ai_quarantine_plan_actions)
    print(f"AI-Quarantaene-PLAN-Report: {ai_quar_plan_path}")
else:
    print("Kein AI-Quarantaene-PLAN-Report geschrieben (keine geplanten Verschiebungen).")

if ai_quarantine_exec_actions:
    save_csv(ai_quar_exec_path, ai_quarantine_exec_actions)
    print(f"AI-Quarantaene-EXEC-Report: {ai_quar_exec_path}")
else:
    print("Kein AI-Quarantaene-EXEC-Report geschrieben (nichts ausgefuehrt).")


[19:12:41] ===== PHASE 6B: KI REPORTS SCHREIBEN =====
AI-Visual-PLAN-Report: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\ai_visual_groups_plan_20260301_191241.csv
AI-Visual-PAIR-Report: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\ai_visual_groups_pairs_20260301_191241.csv
AI-Quarantaene-PLAN-Report: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\ai_quarantine_plan_20260301_191241.csv
Kein AI-Quarantaene-EXEC-Report geschrieben (nichts ausgefuehrt).


## Verwendung
1. `ROOT_DIR` setzen.
2. Mit `DRY_RUN=True` zuerst PHASE 1, 1B, 1C, 1D, 3 laufen lassen (nur Vorschau).
3. PHASE 1D zeigt: pro KI-Gruppe 1 behalten, Rest in Quarantaene verschieben.
4. Wenn ok: `DRY_RUN=False`, `CONFIRM_EXECUTION=True`, `CONFIRM_TEXT="JA_LOESCHEN_UND_UMBENENNEN"`.
5. Fuer KI-Quarantaene zusaetzlich `MOVE_AI_GROUPS_TO_QUARANTINE=True` setzen und PHASE 1D starten.
6. Reports in PHASE 6/6B schreiben.

Hinweis: Quarantaene verschiebt nur in `_duplicate_cleaner_quarantine/ai_visual` und loescht nichts direkt.
